# CRISP-DM: Telco Customer Churn

This notebook uses the uploaded `WA_Fn-UseC_-Telco-Customer-Churn.csv` dataset and demonstrates Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, and Deployment.

**Decision:** prioritize likely churners for human-reviewed retention support.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
from pathlib import Path
import json, warnings, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); RANDOM_STATE=42

## Phase 1 — Business Understanding

The telecom provider wants to reduce preventable churn while controlling the cost of retention offers. The data-mining goal is to estimate churn probability. Success means comparing a baseline and predictive models, reporting class-appropriate metrics, selecting an explicit operating threshold, and saving a reusable model artifact. Customer scores must not automatically deny service.

## Phase 2 — Data Understanding

Load the uploaded file, inspect its structure, assess data quality, and explore churn by contract, internet service, tenure, and charges.

In [ ]:
filename="WA_Fn-UseC_-Telco-Customer-Churn.csv"
candidates=[Path(filename),Path("/content")/filename,Path("upload")/filename,Path("/workspace/scratch/16abedd5f8ac/upload")/filename,Path("projects/crisp_dm/artifacts/telco.csv")]
data_path=next((p for p in candidates if p.exists()),None)
if data_path is None:
    try:
        from google.colab import files
        uploaded=files.upload(); data_path=Path(next(iter(uploaded)))
    except Exception as exc:
        raise FileNotFoundError(f"Upload {filename} and rerun this cell") from exc
df=pd.read_csv(data_path)
print("Using:",data_path,"Shape:",df.shape)
display(df.head())

In [ ]:
profile=pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum(),"unique":df.nunique(dropna=False)})
display(profile)
print("Duplicate rows:",df.duplicated().sum())
display(df["Churn"].value_counts().rename_axis("Churn").to_frame("count"))
print("TotalCharges raw type:",df["TotalCharges"].dtype)
print("Blank TotalCharges:",(df["TotalCharges"].astype(str).str.strip()=="").sum())

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(17,4))
sns.countplot(data=df,x="Churn",ax=axes[0],palette="Set2")
sns.countplot(data=df,x="Contract",hue="Churn",ax=axes[1])
sns.countplot(data=df,x="InternetService",hue="Churn",ax=axes[2])
axes[0].set_title("Churn distribution"); axes[1].set_title("Churn by contract"); axes[2].set_title("Churn by internet service")
axes[1].tick_params(axis="x",rotation=20); axes[2].tick_params(axis="x",rotation=20); plt.tight_layout(); plt.show()
print("Churn rate:",round(df["Churn"].eq("Yes").mean()*100,2),"%")

In [ ]:
eda=df.copy(); eda["TotalCharges"]=pd.to_numeric(eda["TotalCharges"],errors="coerce")
display(eda.groupby("Churn")[["tenure","MonthlyCharges","TotalCharges"]].agg(["mean","median"]).round(2))
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,col in zip(axes,["tenure","MonthlyCharges","TotalCharges"]): sns.boxplot(data=eda,x="Churn",y=col,ax=ax); ax.set_title(f"{col} by churn")
plt.tight_layout(); plt.show()

## Phase 3 — Data Preparation

Convert `TotalCharges` to numeric, convert `Churn` to 0/1, remove `customerID`, use a stratified split, and fit imputation, scaling, and one-hot encoding only on training data to prevent leakage.

In [ ]:
prepared=df.copy(); prepared["TotalCharges"]=pd.to_numeric(prepared["TotalCharges"],errors="coerce"); prepared["Churn"]=prepared["Churn"].eq("Yes").astype(int)
X=prepared.drop(columns=["Churn","customerID"]); y=prepared["Churn"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
nums=X.select_dtypes(include=["number"]).columns.tolist(); cats=X.select_dtypes(exclude=["number"]).columns.tolist()
pre=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),nums),("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),cats)])
print("Train:",X_train.shape,"Test:",X_test.shape)

## Phase 4 — Modeling

Compare a majority-class baseline, balanced logistic regression, and balanced random forest. Logistic regression is retained as the primary deployment candidate because its probability score is easier to audit.

In [ ]:
def pipe(model): return Pipeline([("preprocessor",pre),("model",model)])
models={"baseline":pipe(DummyClassifier(strategy="most_frequent")),"logistic_regression":pipe(LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE)),"random_forest":pipe(RandomForestClassifier(n_estimators=300,min_samples_leaf=3,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1))}
fitted={}; probs={}; preds={}
for name,model in models.items():
    model.fit(X_train,y_train); fitted[name]=model; probs[name]=model.predict_proba(X_test)[:,1]; preds[name]=(probs[name]>=.5).astype(int)
    print("Trained:",name)

## Phase 5 — Evaluation

Use ROC-AUC and average precision for ranking, plus precision, recall, F1, accuracy, and a confusion matrix. Precision estimates the quality of the retention queue; recall estimates how many churners are found.

In [ ]:
def evaluate(name): return {"model":name,"accuracy":accuracy_score(y_test,preds[name]),"precision":precision_score(y_test,preds[name],zero_division=0),"recall":recall_score(y_test,preds[name],zero_division=0),"f1":f1_score(y_test,preds[name],zero_division=0),"roc_auc":roc_auc_score(y_test,probs[name]),"average_precision":average_precision_score(y_test,probs[name])}
results=pd.DataFrame([evaluate(n) for n in models]).sort_values("roc_auc",ascending=False); display(results.round(4))
selected_name="logistic_regression"; selected_model=fitted[selected_name]; selected_prob=probs[selected_name]
cm=confusion_matrix(y_test,preds[selected_name]); sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False); plt.title("Logistic regression confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()
print(classification_report(y_test,preds[selected_name],target_names=["No churn","Churn"]))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
for name in models:
    fpr,tpr,_=roc_curve(y_test,probs[name]); precision,recall,_=precision_recall_curve(y_test,probs[name])
    axes[0].plot(fpr,tpr,label=f"{name} AUC={roc_auc_score(y_test,probs[name]):.3f}"); axes[1].plot(recall,precision,label=name)
axes[0].plot([0,1],[0,1],"k--",alpha=.5); axes[0].set(title="ROC curves",xlabel="False-positive rate",ylabel="True-positive rate"); axes[1].set(title="Precision-recall curves",xlabel="Recall",ylabel="Precision")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
precision,recall,thresholds=precision_recall_curve(y_test,selected_prob)
valid=np.where((precision[:-1]>=.65)&(recall[:-1]>=.35))[0]
business_threshold=float(thresholds[valid[0]]) if len(valid) else .50
business_pred=(selected_prob>=business_threshold).astype(int)
threshold_metrics={"threshold":business_threshold,"precision":precision_score(y_test,business_pred,zero_division=0),"recall":recall_score(y_test,business_pred,zero_division=0),"f1":f1_score(y_test,business_pred,zero_division=0),"flagged_customers":int(business_pred.sum())}
display(pd.DataFrame([threshold_metrics]).round(4))

## Phase 6 — Deployment

Save the complete preprocessing-plus-model pipeline and demonstrate scoring one new customer. Production deployment should add authentication, input validation, version logging, calibration, drift monitoring, fairness checks, retraining, and human approval.

In [ ]:
artifact_dir=Path("crisp_dm_artifacts"); artifact_dir.mkdir(exist_ok=True)
metrics={"dataset":str(data_path),"rows":int(len(df)),"columns":int(df.shape[1]),"churn_rate":float(y.mean()),"selected_model":selected_name,"model_results":results.to_dict(orient="records"),"business_threshold":threshold_metrics,"deployment_action":"Send customers above the threshold to a human-reviewed retention queue."}
(artifact_dir/"metrics.json").write_text(json.dumps(metrics,indent=2)); joblib.dump(selected_model,artifact_dir/"telco_churn_logistic_pipeline.joblib"); results.to_csv(artifact_dir/"model_comparison.csv",index=False)
new_customer=pd.DataFrame([{"gender":"Female","SeniorCitizen":0,"Partner":"Yes","Dependents":"No","tenure":4,"PhoneService":"Yes","MultipleLines":"No","InternetService":"Fiber optic","OnlineSecurity":"No","OnlineBackup":"No","DeviceProtection":"No","TechSupport":"No","StreamingTV":"Yes","StreamingMovies":"Yes","Contract":"Month-to-month","PaperlessBilling":"Yes","PaymentMethod":"Electronic check","MonthlyCharges":85.0,"TotalCharges":340.0}])
risk=float(selected_model.predict_proba(new_customer)[:,1][0]); print(f"Predicted churn probability: {risk:.2%}"); print("Decision:","Human review for retention support" if risk>=business_threshold else "Routine service monitoring")
print("Saved:",[p.name for p in artifact_dir.iterdir()])

## Final conclusion and manual verification checklist

This project connects the retention problem to data profiling, leakage-safe preparation, model comparison, threshold-based evaluation, and a saved deployment artifact. Before submission, verify the uploaded CSV, target definition, excluded identifier, numeric conversion of `TotalCharges`, training-only preprocessing, untouched test set, threshold rule, saved artifacts, and screenshots of successful execution.